In [0]:
spark.table(
    "workspace.transportation_analytics.gold_route_profitability"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

silver_trips = spark.table(
    "workspace.transportation_analytics.silver_trips"
)

silver_fuel_purchases = spark.table(
    "workspace.transportation_analytics.silver_fuel_purchases"
)

print("Silver route profitability tables loaded successfully")

silver_routes.printSchema()
silver_loads.printSchema()
silver_trips.printSchema()
silver_fuel_purchases.printSchema()

In [0]:
gold_route_profitability = (
    silver_loads
    .join(
        silver_trips.select(
            "trip_id",
            "load_id",
            "actual_distance_miles"
        ),
        on="load_id",
        how="left"
    )
    .join(
        silver_fuel_purchases.select(
            "trip_id",
            "total_cost"
        ),
        on="trip_id",
        how="left"
    )
)

print("Loads, trips, and fuel costs joined successfully")

In [0]:
gold_route_profitability = (
    gold_route_profitability
    .groupBy("route_id")
    .agg(
        F.countDistinct("load_id").alias("total_loads"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("total_cost").alias("total_fuel_cost"),
        F.sum("actual_distance_miles").alias("total_distance_miles")
    )
)

print("Route-level profitability metrics calculated successfully")

In [0]:
gold_route_profitability = (
    gold_route_profitability
    .withColumn(
        "profit",
        F.round(
            F.col("total_revenue") - F.col("total_fuel_cost"),
            2
        )
    )
    .withColumn(
        "profit_per_mile",
        F.round(
            F.col("profit") / F.col("total_distance_miles"),
            2
        )
    )
)

print("Route profitability calculated successfully")

In [0]:
gold_route_profitability = (
    gold_route_profitability
    .join(
        silver_routes.select(
            "route_id",
            "origin_city",
            "origin_state",
            "destination_city",
            "destination_state",
            "typical_distance_miles"
        ),
        on="route_id",
        how="left"
    )
)

print("Route details added to Gold profitability data")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_route_profitability"
)

target.alias("t").merge(
    gold_route_profitability.alias("s"),
    "t.route_id = s.route_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Route Profitability table updated using MERGE")

In [0]:
spark.table(
    "workspace.transportation_analytics.gold_route_profitability"
).show(10, truncate=False)

In [0]:
display(
    gold_route_profitability
    .orderBy(F.desc("profit"))
    .select(
        "route_id",
        "total_loads",
        "total_revenue",
        "total_fuel_cost",
        "profit",
        "profit_per_mile"
    )
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
print("MOST PROFITABLE ROUTES")
gold_route_profitability \
    .orderBy(F.desc("profit")) \
    .select(
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "profit",
        "profit_per_mile"
    ) \
    .show(5, truncate=False)

print("LEAST PROFITABLE ROUTES")
gold_route_profitability \
    .orderBy(F.asc("profit")) \
    .select(
        "route_id",
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state",
        "profit",
        "profit_per_mile"
    ) \
    .show(5, truncate=False)